# KinoCompass – итоговый тест: Mult-VAE и смесь с ним

**Тест – один раз.** Модели и настройки выбраны на валидации и зафиксированы в репозитории (шаг 6в); здесь они только переобучаются на обучении + валидации (всё до 14.10.2022) и проверяются на последнем годе (14.10.2022 – 13.10.2023). Повторно не запускать и ничего не подбирать по результату.

Состав (`python evaluate.py final_gpu`): популярные за последний год (для парных сравнений), Mult-VAE («вкус» 400, β 0,5) и смесь «популярность × Mult-VAE» с весом 0,2. EASE – отдельным ноутбуком `final_ease.ipynb` (на тесте ему трижды не хватило памяти, и он не должен срывать этот прогон). Остальные модели – в Docker (`python evaluate.py final`): те же люди и сценарии, модели сравниваются парным тестом между прогонами.

**Как запустить:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**, затем «Среда выполнения» → «Выполнить все». В конце скачается архив `final_gpu_test.zip` – его нужно положить в `model/reports/`.

Время на T4 (бесплатный Colab): загрузка набора и сценарии – около 2 минут, Mult-VAE – 8 минут (эпоха – 12 с), проверка – 2,5 минуты; всего 13 минут.

In [6]:
# итоговый тест – один раз (модели выбраны на валидации, шаг 6в)
PERIOD = "test"

In [7]:
import torch

assert torch.cuda.is_available(), "нет GPU: Среда выполнения -> Сменить среду выполнения -> T4 GPU"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "ГБ")

Tesla T4 14.6 ГБ


In [8]:
# из /content: при повторном запуске в том же сеансе текущая папка – внутри удаляемого клона
%cd /content
!rm -rf /content/KinoCompass && git clone -q --depth 1 https://github.com/AlexTrav/KinoCompass.git /content/KinoCompass
!git -C /content/KinoCompass log --oneline -1
# те же версии, что в образе Docker (torch в Colab уже есть)
!pip install -q requests==2.34.2 pandas==3.1.0 pyarrow==26.0.1 scipy==1.18.1

/content
07beafc (grafted, HEAD -> main, origin/main, origin/HEAD) Разделил итоговый тест в Colab: Mult-VAE отдельно, EASE полосами на GPU с проверкой памяти


In [9]:
%cd /content/KinoCompass/model
!PYTHONPATH=/content/KinoCompass/data:/content/KinoCompass/research PERIOD={PERIOD} python -W ignore evaluate.py final_gpu

/content/KinoCompass/model
скачиваю https://files.grouplens.org/datasets/movielens/ml-32m.zip в память
  0 % (1 из 227 МБ)
  25 % (57 из 227 МБ)
  50 % (114 из 227 МБ)
  75 % (171 из 227 МБ)
  100 % (227 из 227 МБ)
  ratings: 32 000 204 строк
  tags: 2 000 072 строк
  movies: 87 585 строк
  links: 87 585 строк
кандидаты: candidates, 22309 фильмов
сценарии: {'cold5': {'people': 4258, 'median_inputs': 5, 'median_targets': 44}, 'cold10': {'people': 4252, 'median_inputs': 10, 'median_targets': 41}, 'warm': {'people': 5413, 'median_inputs': 320, 'median_targets': 8}}
cold5 popular_year ndcg@10 0.28512
cold10 popular_year ndcg@10 0.24941
warm popular_year ndcg@10 0.09724
vae_z400_b0.5: эпоха 1/40, потеря 1234.434, 11.5 с
vae_z400_b0.5: эпоха 10/40, потеря 1128.998, 12.1 с
vae_z400_b0.5: эпоха 20/40, потеря 1130.085, 12.0 с
vae_z400_b0.5: эпоха 30/40, потеря 1126.361, 12.0 с
vae_z400_b0.5: эпоха 40/40, потеря 1123.902, 12.0 с
cold5 hybrid_popular_year_vae_z400_b0.5_w0.2 ndcg@10 0.3771
cold10 

In [10]:
from google.colab import files

!cd /content/KinoCompass/model && zip -q -r /content/final_gpu_{PERIOD}.zip reports/final_gpu && ls -la reports/final_gpu
files.download(f"/content/final_gpu_{PERIOD}.zip")

total 132
drwxr-xr-x  2 root root   4096 Nov 11 11:15 .
drwxr-xr-x 13 root root   4096 Nov 11 11:15 ..
-rw-r--r--  1 root root   6952 Nov 11 11:15 test.json
-rw-r--r--  1 root root 116572 Nov 11 11:15 test_users.csv.gz
